# 05 · Register caps and pipelining depth

Two experiments on the 4096³ matmul:

* **`maxnreg`**: cap the registers ptxas may use per thread. Fewer registers per thread lets more
  blocks fit on an SM, but anything that no longer fits is spilled to local memory.
* **`num_stages`**: change the software-pipelining depth. Each stage adds a shared-memory buffer
  (filled with `cp.async` on Ampere), which hides memory latency until shared memory starts to
  limit how many blocks fit.

In [ ]:
# Setup: mount Drive, make sure the pinned torch/triton are installed, import archtune.
import importlib.metadata as md, os, subprocess, sys
PROJECT = "/content/drive/MyDrive/archtune"
NEED_GPU_STACK = True

if not os.path.ismount("/content/drive"):
    from google.colab import drive
    drive.mount("/content/drive")
os.chdir(PROJECT)
sys.path.insert(0, PROJECT)

def _ver(pkg):
    try:
        return md.version(pkg).split("+")[0]
    except md.PackageNotFoundError:
        return None

if NEED_GPU_STACK and (_ver("torch"), _ver("triton")) != ("2.6.0", "3.2.0"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch==2.6.0", "triton==3.2.0"], check=True)
    if "torch" in sys.modules:
        raise SystemExit("torch was already imported: restart the session and run this cell again")

from archtune import config, gpu, model, compiler, run
ENV = gpu.capture()
MINI = config.mini_mode()
RESULTS = os.environ.get("ARCHTUNE_RESULTS") or os.path.join(PROJECT, "results_mini" if MINI else "results")
print(ENV["gpu_name"], "| torch", ENV["torch_version"], "| triton", ENV["triton_version"], "|", RESULTS)
if NEED_GPU_STACK and not gpu.pinned_ok(ENV):
    raise SystemExit("wrong torch/triton loaded: restart the session and run this cell again")

In [ ]:
DIMS = model.derive_dims(model.load_model_config(cache_dir=RESULTS))
MM_SHAPES = model.matmul_shapes(DIMS)
RMS_SHAPES = model.rmsnorm_shapes(DIMS)
SPLIT = {**model.train_test_split([s.shape_id for s in MM_SHAPES]),
         **model.train_test_split([s.shape_id for s in RMS_SHAPES])}
if MINI:
    small = model.mini_shapes(MM_SHAPES + RMS_SHAPES)
    MM_SHAPES = [s for s in small if s.kernel == "matmul"]
    RMS_SHAPES = [s for s in small if s.kernel == "rmsnorm"]
print(DIMS)
for s in MM_SHAPES + RMS_SHAPES:
    print(f"{s.shape_id:24s} {SPLIT[s.shape_id]:5s} {','.join(s.roles)}")

In [ ]:
import os, pandas as pd
EX = config.EXPERIMENTS
GPU_DIR = os.path.join(RESULTS, ENV["gpu_tag"])
SW = run.load_sweep(RESULTS, ENV["gpu_tag"])
SHAPE = next(s for s in MM_SHAPES if s.shape_id == EX.large_shape_id)
supported, why = gpu.maxnreg_supported(); print(why)

In [ ]:
if supported:
    reps = run.representative_configs(SW, SHAPE.shape_id)
    run.run_sweep(run.make_tasks([SHAPE], lambda s: run.with_maxnreg(reps)), RESULTS, ENV,
                  run_id=EX.maxnreg_run_id, table="experiments_maxnreg")
    frame = run.maxnreg_frame(run.consolidate(GPU_DIR, "experiments_maxnreg"), gpu.live())
    display(frame)
    run.fig3_maxnreg(frame, f"Register cap sweep — {ENV['gpu_tag']}, {SHAPE.shape_id}",
                     os.path.join(RESULTS, "figures", f"fig3_maxnreg_{ENV['gpu_tag']}.png"))

In [ ]:
v = model.prepare(SW); v = v[v.valid & (v.kernel == "matmul")]
shapes = {s.shape_id: s for s in MM_SHAPES}
tasks = []
for sid in dict.fromkeys([EX.large_shape_id, sorted(v.shape_id.unique())[0]]):
    for _, r in v[v.shape_id == sid].nsmallest(EX.stages_top_configs, "time_ms_median").iterrows():
        tasks += run.make_tasks([shapes[sid]], lambda s, c=model.row_config(r): run.with_num_stages([c]))
run.run_sweep(tasks, RESULTS, ENV, run_id=EX.stages_run_id, table="experiments_stages")
frame4 = run.num_stages_frame(run.consolidate(GPU_DIR, "experiments_stages"))
display(frame4.groupby("num_stages")[["shared_bytes", "ptx_cp_async", "n_regs", "time_rel"]].median())
run.fig4_num_stages(frame4, f"Pipelining depth — {ENV['gpu_tag']}",
                    os.path.join(RESULTS, "figures", f"fig4_num_stages_{ENV['gpu_tag']}.png"))